In [1]:
from pathlib import Path
import gc
import json
import random
import time
import warnings
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import SelectKBest, chi2
from sklearn.metrics import accuracy_score, balanced_accuracy_score, confusion_matrix, f1_score, matthews_corrcoef, precision_score, recall_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import MinMaxScaler, Normalizer, OneHotEncoder, OrdinalEncoder, StandardScaler
import tensorflow as tf
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Activation, BatchNormalization, Conv1D, Dense, Dropout, Flatten, Input, MaxPooling1D
from tensorflow.keras.optimizers import Adam
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset


In [3]:
from os import name
column_names = ['duration', 'protocol_type', 'service', 'flag', 'src_bytes', 'dst_bytes', 'land', 'wrong_fragment', 'urgent', 'hot', 'num_failed_logins', 'logged_in', 'num_compromised', 'root_shell', 'su_attempted', 'num_root', 'num_file_creations', 'num_shells', 'num_access_files', 'num_outbound_cmds', 'is_host_login', 'is_guest_login', 'count', 'srv_count', 'serror_rate', 'srv_serror_rate', 'rerror_rate', 'srv_rerror_rate', 'same_srv_rate', 'diff_srv_rate', 'srv_diff_host_rate', 'dst_host_count', 'dst_host_srv_count', 'dst_host_same_srv_rate', 'dst_host_diff_srv_rate', 'dst_host_same_src_port_rate', 'dst_host_srv_diff_host_rate', 'dst_host_serror_rate', 'dst_host_srv_serror_rate', 'dst_host_rerror_rate', 'dst_host_srv_rerror_rate', 'label', 'difficulty']


In [5]:
from pathlib import Path
from sklearn.model_selection import GroupShuffleSplit
RANDOM_STATE = 42
PREPARATION_SEED = 42
VALIDATION_SIZE = 0.2
SOURCE_COLUMN = 'source_pcap'
LABEL_COLUMN = 'label'
BINARY_COLUMN = 'binary_label'
FEATURE_COLUMNS = column_names[:41]
CATEGORICAL_COLUMNS = ['protocol_type', 'service', 'flag']
MODEL_KEYS = ['kwon_cnn', 'vinayakumar_dnn', 'kilichev_cnn', 'rf_selected_knn', 'ckan']
SEEDS = [11, 22, 33, 44, 55]
MAX_TRAIN_ROWS = None
MAX_TEST_ROWS = None
EPOCH_OVERRIDES = {'kwon_cnn': 10, 'vinayakumar_dnn': 20, 'kilichev_cnn': 72, 'ckan': 20}
BATCH_SIZE_OVERRIDES = {'kwon_cnn': 32, 'vinayakumar_dnn': 64, 'kilichev_cnn': 32, 'rf_selected_knn': 1024, 'ckan': 32}
TARGET_TEST_NORMAL_FRACTION = 0.431
TARGET_TEST_ATTACK_FRACTION = 0.569
EXP4_EXPERIMENT = 'mixed_source_nsl_kdd'
OUTPUT_DIR = Path('nsl_kdd_prepared')
RESULTS_DIR = Path('nsl_kdd_results')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
SAVE_RATIO_MATCHED_TEST_SETS = True
print('Target: 43.1% normal / 56.9% attack')
print('Results:', RESULTS_DIR.resolve())


Target: 43.1% normal / 56.9% attack
Results: /content/nsl_kdd_results


In [6]:
FEATURE_COLUMNS = column_names[:41]

def normalize_label(value):
    if pd.isna(value):
        return pd.NA
    return str(value).strip().rstrip('.').casefold()

def make_binary_target(frame):
    labels = frame[LABEL_COLUMN].map(normalize_label)
    return np.where(labels.eq('normal') | labels.eq('benign'), 0, 1).astype(np.int32)

def validate_frame(frame, name):
    missing = sorted(set(FEATURE_COLUMNS + [LABEL_COLUMN]) - set(frame.columns))
    if missing:
        raise KeyError(f'{name} is missing required columns: {missing}')

def downsample_frame(frame, max_rows, seed):
    if max_rows is None or len(frame) <= max_rows:
        return frame.reset_index(drop=True)
    target = make_binary_target(frame)
    sampled, _ = train_test_split(frame, train_size=max_rows, random_state=seed, stratify=target)
    return sampled.reset_index(drop=True)


In [7]:
def validate_columns(frame, required, frame_name):
    missing = sorted(set(required) - set(frame.columns))
    if missing:
        raise KeyError(f'{frame_name} is missing columns: {missing}')

def ensure_binary_label(frame):
    result = frame.copy()
    result[BINARY_COLUMN] = np.where(result[LABEL_COLUMN].eq('benign'), 'benign', 'attack')
    return result

def source_label_distribution(frame, label_column=LABEL_COLUMN):
    table = frame.groupby([SOURCE_COLUMN, label_column], dropna=False).size().reset_index(name='count')
    table['within_source_pct'] = 100 * table['count'] / table.groupby(SOURCE_COLUMN)['count'].transform('sum')
    return table.sort_values([SOURCE_COLUMN, 'count'], ascending=[True, False])

def _largest_remainder_allocation(counts, requested_total):
    """Allocate a sample across strata while preserving proportions."""
    counts = pd.Series(counts, dtype='int64')
    available_total = int(counts.sum())
    if requested_total < 0 or requested_total > available_total:
        raise ValueError(f'requested_total={requested_total} exceeds available_total={available_total}')
    if requested_total == 0:
        return counts * 0
    raw = counts / available_total * requested_total
    allocation = np.floor(raw).astype('int64')
    remaining = int(requested_total - allocation.sum())
    fractional = (raw - allocation).sort_values(ascending=False)
    while remaining > 0:
        progressed = False
        for stratum in fractional.index:
            if allocation.loc[stratum] < counts.loc[stratum]:
                allocation.loc[stratum] += 1
                remaining -= 1
                progressed = True
                if remaining == 0:
                    break
        if not progressed:
            raise RuntimeError('Could not complete proportional allocation.')
    return allocation

def proportional_stratified_sample(frame, requested_total, strata_column, random_state):
    if requested_total == len(frame):
        return frame.copy()
    strata = frame[strata_column].astype('string').fillna('<missing>')
    counts = strata.value_counts(sort=False)
    allocation = _largest_remainder_allocation(counts, requested_total)
    sampled_parts = []
    ordered_strata = sorted(allocation.index.astype(str))
    for offset, stratum in enumerate(ordered_strata):
        take = int(allocation.loc[stratum])
        if take == 0:
            continue
        cell = frame.loc[strata.eq(stratum)]
        sampled_parts.append(cell.sample(n=take, replace=False, random_state=random_state + offset))
    if not sampled_parts:
        return frame.iloc[0:0].copy()
    return pd.concat(sampled_parts, axis=0)

def match_binary_test_ratio(frame, target_attack_fraction=TARGET_TEST_ATTACK_FRACTION, random_state=PREPARATION_SEED):
    if not 0.0 < target_attack_fraction < 1.0:
        raise ValueError('target_attack_fraction must be between 0 and 1.')
    working = frame.copy()
    working['__ratio_original_order'] = np.arange(len(working))
    working['__ratio_binary'] = make_binary_target(working)
    attack_rows = working[working['__ratio_binary'].eq(1)]
    normal_rows = working[working['__ratio_binary'].eq(0)]
    available_attack = int(len(attack_rows))
    available_normal = int(len(normal_rows))
    if available_attack == 0 or available_normal == 0:
        raise ValueError('Every ratio-matched test source must contain both attack and normal rows.')
    target_normal_fraction = 1.0 - target_attack_fraction
    maximum_total = min(int(np.floor(available_attack / target_attack_fraction)), int(np.floor(available_normal / target_normal_fraction)))
    while maximum_total > 1:
        selected_attack = int(round(maximum_total * target_attack_fraction))
        selected_normal = maximum_total - selected_attack
        if 0 < selected_attack <= available_attack and 0 < selected_normal <= available_normal:
            break
        maximum_total -= 1
    else:
        raise ValueError('Not enough rows to construct a matched test set.')
    sampled_attack = proportional_stratified_sample(attack_rows, selected_attack, LABEL_COLUMN, random_state)
    sampled_normal = normal_rows.sample(n=selected_normal, replace=False, random_state=random_state + 10000)
    matched = pd.concat([sampled_attack, sampled_normal], axis=0).sort_values('__ratio_original_order').drop(columns=['__ratio_original_order', '__ratio_binary']).reset_index(drop=True)
    actual_attack_fraction = float(make_binary_target(matched).mean())
    audit = {'available_rows': int(len(frame)), 'available_attack': available_attack, 'available_normal': available_normal, 'selected_rows': int(len(matched)), 'selected_attack': int(selected_attack), 'selected_normal': int(selected_normal), 'target_attack_pct': 100.0 * target_attack_fraction, 'target_normal_pct': 100.0 * target_normal_fraction, 'actual_attack_pct': 100.0 * actual_attack_fraction, 'actual_normal_pct': 100.0 * (1.0 - actual_attack_fraction)}
    if round(audit['actual_normal_pct'], 1) != round(audit['target_normal_pct'], 1):
        raise AssertionError(f'The matched test set does not round to the requested normal prevalence: {audit}')
    return (matched, audit)

def ratio_match_test_sets(test_sets, target_attack_fraction=TARGET_TEST_ATTACK_FRACTION, random_state=PREPARATION_SEED):
    matched_sets = {}
    audit_rows = []
    for offset, (test_source, test_frame) in enumerate(test_sets.items()):
        matched, audit = match_binary_test_ratio(test_frame, target_attack_fraction=target_attack_fraction, random_state=random_state + 1000 * offset)
        matched['ratio_match'] = '43.1% normal / 56.9% attack'
        matched_sets[test_source] = matched
        audit_rows.append({'test_source': test_source, **audit})
    return (matched_sets, pd.DataFrame(audit_rows))


In [ ]:
PARTITION_DIR = Path('/content/drive/MyDrive/Array_submission/Prepared_partitions/splits') / 'kdd_nsl'
exp4_train_balanced = ensure_binary_label(pd.read_csv(PARTITION_DIR / 'balanced_train.csv', low_memory=False))
exp4_external_train_balanced = exp4_train_balanced
exp4_external_test_natural = ensure_binary_label(pd.read_csv(PARTITION_DIR / 'test_all_sources.csv', low_memory=False))
validate_columns(exp4_train_balanced, set(FEATURE_COLUMNS + [SOURCE_COLUMN, LABEL_COLUMN, BINARY_COLUMN]), 'balanced_train')
validate_columns(exp4_external_test_natural, set(FEATURE_COLUMNS + [SOURCE_COLUMN, LABEL_COLUMN, BINARY_COLUMN]), 'test_all_sources')
exp4_natural_tests = {source: part.reset_index(drop=True) for source, part in exp4_external_test_natural.groupby(SOURCE_COLUMN, sort=True)}
exp4_test_sets, exp4_test_ratio_manifest = ratio_match_test_sets(exp4_natural_tests, target_attack_fraction=TARGET_TEST_ATTACK_FRACTION, random_state=PREPARATION_SEED)
print('Loaded balanced training set:', exp4_train_balanced.shape)
print('Loaded test partition:', exp4_external_test_natural.shape)

Loaded balanced training set: (65120, 47)
Loaded test partition: (27912, 47)


In [ ]:
partition_manifest = pd.read_csv(PARTITION_DIR / 'partition_manifest.csv')
training_distribution = pd.read_csv(PARTITION_DIR / 'balanced_training_distribution.csv')
test_distribution = pd.read_csv(PARTITION_DIR / 'test_distribution.csv')
exp4_test_ratio_manifest.to_csv(RESULTS_DIR / f'{EXP4_EXPERIMENT}_test_ratio_manifest.csv', index=False)
training_distribution.to_csv(RESULTS_DIR / 'balanced_training_distribution.csv', index=False)
test_distribution.to_csv(RESULTS_DIR / 'test_distribution_before_ratio_matching.csv', index=False)
for source, frame in exp4_test_sets.items():
    frame.to_csv(RESULTS_DIR / f'{EXP4_EXPERIMENT}_test_{source}.csv', index=False)
pd.concat(exp4_test_sets.values(), ignore_index=True).to_csv(RESULTS_DIR / f'{EXP4_EXPERIMENT}_test_all_sources.csv', index=False)


In [12]:
def calculate_binary_metrics(y_true, y_pred, y_probability=None):
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()
    result = {'n_test': int(len(y_true)), 'accuracy': accuracy_score(y_true, y_pred), 'balanced_accuracy': balanced_accuracy_score(y_true, y_pred), 'precision': precision_score(y_true, y_pred, zero_division=0), 'recall': recall_score(y_true, y_pred, zero_division=0), 'specificity': tn / (tn + fp) if tn + fp else np.nan, 'f1': f1_score(y_true, y_pred, zero_division=0), 'macro_f1': f1_score(y_true, y_pred, average='macro', zero_division=0), 'mcc': matthews_corrcoef(y_true, y_pred), 'false_positive_rate': fp / (fp + tn) if fp + tn else np.nan, 'tn': int(tn), 'fp': int(fp), 'fn': int(fn), 'tp': int(tp)}
    if y_probability is not None and len(np.unique(y_true)) == 2:
        result['roc_auc'] = roc_auc_score(y_true, y_probability)
    else:
        result['roc_auc'] = np.nan
    return result

def category_recall_table(frame, y_true, y_pred):
    labels = frame[LABEL_COLUMN].astype(str).to_numpy()
    rows = []
    for category in sorted(pd.unique(labels)):
        mask = labels == category
        category_true = y_true[mask]
        category_pred = y_pred[mask]
        rows.append({'test_category': category, 'support': int(mask.sum()), 'binary_attack_rate': float(category_true.mean()), 'binary_recall': recall_score(category_true, category_pred, pos_label=1, zero_division=0) if np.any(category_true == 1) else np.nan, 'benign_specificity': recall_score(category_true, category_pred, pos_label=0, zero_division=0) if np.any(category_true == 0) else np.nan})
    return pd.DataFrame(rows)

def set_all_seeds(seed):
    random.seed(seed)
    np.random.seed(seed)
    tf.keras.utils.set_random_seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


In [13]:
def _prepare_test_frame(test_frame, seed):
    return downsample_frame(test_frame, MAX_TEST_ROWS, seed)

def _keras_binary_prediction(model, X, batch_size):
    probabilities = model.predict(X, batch_size=batch_size, verbose=0).reshape(-1)
    predictions = (probabilities >= 0.5).astype(np.int32)
    return (predictions, probabilities)


In [14]:
def fit_kwon(train_frame, seed):
    set_all_seeds(seed)
    tf.keras.backend.clear_session()
    train_frame = downsample_frame(train_frame, MAX_TRAIN_ROWS, seed)
    X_train_df = train_frame[FEATURE_COLUMNS].copy()
    y_train = make_binary_target(train_frame)
    numerical_columns = [column for column in FEATURE_COLUMNS if column not in CATEGORICAL_COLUMNS]
    preprocessor = ColumnTransformer(transformers=[('numerical', StandardScaler(), numerical_columns), ('categorical', OneHotEncoder(handle_unknown='ignore', sparse_output=False, dtype=np.float32), CATEGORICAL_COLUMNS)], sparse_threshold=0.0)
    print('    Fitting preprocessing...', flush=True)
    X_train = preprocessor.fit_transform(X_train_df).astype(np.float32)
    X_train = np.expand_dims(X_train, axis=-1)
    model = Sequential([Input(shape=(X_train.shape[1], 1)), Conv1D(64, 3, strides=1, padding='same', use_bias=False), Activation('relu'), BatchNormalization(), Dropout(0.5), MaxPooling1D(pool_size=2, strides=2), Flatten(), Dense(64, use_bias=False), Activation('relu'), BatchNormalization(), Dropout(0.5), Dense(2, activation='softmax')])
    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    print('    Training Kwon CNN...', flush=True)
    model.fit(X_train, y_train, epochs=EPOCH_OVERRIDES['kwon_cnn'], batch_size=BATCH_SIZE_OVERRIDES['kwon_cnn'], shuffle=True, verbose=1)
    return {'model': model, 'preprocessor': preprocessor, 'seed': seed, 'n_train': len(train_frame)}

def predict_kwon(fitted, test_frame):
    test_frame = _prepare_test_frame(test_frame, fitted['seed'])
    X_test = fitted['preprocessor'].transform(test_frame[FEATURE_COLUMNS]).astype(np.float32)
    X_test = np.expand_dims(X_test, axis=-1)
    probabilities = fitted['model'].predict(X_test, batch_size=BATCH_SIZE_OVERRIDES['kwon_cnn'], verbose=0)
    predictions = np.argmax(probabilities, axis=1).astype(np.int32)
    return (make_binary_target(test_frame), predictions, probabilities[:, 1], test_frame)


In [15]:
def build_vinayakumar_model(input_dim=None):
    if input_dim is None:
        input_dim = len(FEATURE_COLUMNS)
    model = tf.keras.Sequential([tf.keras.layers.Input(shape=(input_dim,), name='nsl_kdd_input'), tf.keras.layers.Dense(units=1024, activation='relu', name='dense_1024'), tf.keras.layers.BatchNormalization(name='batch_norm_1024'), tf.keras.layers.Dropout(rate=0.01, name='dropout_1024'), tf.keras.layers.Dense(units=768, activation='relu', name='dense_768'), tf.keras.layers.BatchNormalization(name='batch_norm_768'), tf.keras.layers.Dropout(rate=0.01, name='dropout_768'), tf.keras.layers.Dense(units=512, activation='relu', name='dense_512'), tf.keras.layers.BatchNormalization(name='batch_norm_512'), tf.keras.layers.Dropout(rate=0.01, name='dropout_512'), tf.keras.layers.Dense(units=256, activation='relu', name='dense_256'), tf.keras.layers.BatchNormalization(name='batch_norm_256'), tf.keras.layers.Dropout(rate=0.01, name='dropout_256'), tf.keras.layers.Dense(units=128, activation='relu', name='dense_128'), tf.keras.layers.BatchNormalization(name='batch_norm_128'), tf.keras.layers.Dropout(rate=0.01, name='dropout_128'), tf.keras.layers.Dense(units=1, activation='sigmoid', name='binary_output')], name='nsl_kdd_binary_dnn')
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.1), loss=tf.keras.losses.BinaryCrossentropy(), metrics=[tf.keras.metrics.BinaryAccuracy(name='accuracy', threshold=0.5), tf.keras.metrics.Precision(name='precision', thresholds=0.5), tf.keras.metrics.Recall(name='recall', thresholds=0.5), tf.keras.metrics.AUC(name='auc')])
    return model

def fit_vinayakumar(train_frame, seed):
    set_all_seeds(seed)
    tf.keras.backend.clear_session()
    train_frame = downsample_frame(train_frame, MAX_TRAIN_ROWS, seed)
    X_all_df = train_frame[FEATURE_COLUMNS].copy()
    y_all = make_binary_target(train_frame).astype(np.float32)
    X_train_df, X_val_df, y_train, y_val = train_test_split(X_all_df, y_all, test_size=VALIDATION_SIZE, random_state=seed, stratify=y_all)
    encoder = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
    X_train_df = X_train_df.copy()
    X_val_df = X_val_df.copy()
    X_train_df.loc[:, CATEGORICAL_COLUMNS] = encoder.fit_transform(X_train_df[CATEGORICAL_COLUMNS])
    X_val_df.loc[:, CATEGORICAL_COLUMNS] = encoder.transform(X_val_df[CATEGORICAL_COLUMNS])
    X_train = X_train_df.astype(np.float32).to_numpy()
    X_val = X_val_df.astype(np.float32).to_numpy()
    normalizer = Normalizer(norm='l2')
    X_train = normalizer.fit_transform(X_train).astype(np.float32)
    X_val = normalizer.transform(X_val).astype(np.float32)
    model = build_vinayakumar_model()
    print('    Training Vinayakumar DNN...', flush=True)
    model.fit(X_train, y_train, validation_data=(X_val, y_val), epochs=EPOCH_OVERRIDES['vinayakumar_dnn'], batch_size=BATCH_SIZE_OVERRIDES['vinayakumar_dnn'], shuffle=True, verbose=1)
    return {'model': model, 'encoder': encoder, 'normalizer': normalizer, 'seed': seed, 'n_train': len(train_frame)}

def predict_vinayakumar(fitted, test_frame):
    test_frame = _prepare_test_frame(test_frame, fitted['seed'])
    X_test_df = test_frame[FEATURE_COLUMNS].copy()
    X_test_df.loc[:, CATEGORICAL_COLUMNS] = fitted['encoder'].transform(X_test_df[CATEGORICAL_COLUMNS])
    X_test = X_test_df.astype(np.float32).to_numpy()
    X_test = fitted['normalizer'].transform(X_test).astype(np.float32)
    predictions, probabilities = _keras_binary_prediction(fitted['model'], X_test, BATCH_SIZE_OVERRIDES['vinayakumar_dnn'])
    return (make_binary_target(test_frame), predictions, probabilities, test_frame)


In [16]:
def fit_kilichev(train_frame, seed):
    set_all_seeds(seed)
    tf.keras.backend.clear_session()
    train_frame = downsample_frame(train_frame, MAX_TRAIN_ROWS, seed)
    y_all = make_binary_target(train_frame)
    retained_features = [feature for feature in FEATURE_COLUMNS if feature != 'num_outbound_cmds']
    X_raw = train_frame[retained_features].copy()
    numerical_columns = [column for column in retained_features if column not in CATEGORICAL_COLUMNS]
    preprocessor = ColumnTransformer(transformers=[('numerical', 'passthrough', numerical_columns), ('categorical', OneHotEncoder(handle_unknown='ignore', sparse_output=False, dtype=np.float32), CATEGORICAL_COLUMNS)], sparse_threshold=0.0)
    print('    Fitting encoding and chi-square selection...', flush=True)
    X_all = preprocessor.fit_transform(X_raw).astype(np.float32)
    selector = SelectKBest(score_func=chi2, k=min(100, X_all.shape[1]))
    X_all = selector.fit_transform(X_all, y_all).astype(np.float32)
    X_train, X_val, y_train, y_val = train_test_split(X_all, y_all, test_size=VALIDATION_SIZE, random_state=seed, stratify=y_all)
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train).astype(np.float32)
    X_val = scaler.transform(X_val).astype(np.float32)
    X_train = np.expand_dims(X_train, axis=-1)
    X_val = np.expand_dims(X_val, axis=-1)
    model = Sequential([Input(shape=(X_train.shape[1], 1)), Conv1D(32, 9, activation='relu'), MaxPooling1D(pool_size=5), Conv1D(64, 9, activation='relu'), MaxPooling1D(pool_size=5), Dropout(0.253378), Flatten(), Dense(128, activation='relu'), Dropout(0.253378), Dense(1, activation='sigmoid')])
    model.compile(loss='binary_crossentropy', optimizer=Adam(learning_rate=0.000471), metrics=['accuracy'])
    print('    Training Kilichev CNN...', flush=True)
    model.fit(X_train, y_train, validation_data=(X_val, y_val), batch_size=BATCH_SIZE_OVERRIDES['kilichev_cnn'], epochs=EPOCH_OVERRIDES['kilichev_cnn'], verbose=1)
    return {'model': model, 'preprocessor': preprocessor, 'selector': selector, 'scaler': scaler, 'retained_features': retained_features, 'seed': seed, 'n_train': len(train_frame)}

def predict_kilichev(fitted, test_frame):
    test_frame = _prepare_test_frame(test_frame, fitted['seed'])
    X_test = fitted['preprocessor'].transform(test_frame[fitted['retained_features']]).astype(np.float32)
    X_test = fitted['selector'].transform(X_test).astype(np.float32)
    X_test = fitted['scaler'].transform(X_test).astype(np.float32)
    X_test = np.expand_dims(X_test, axis=-1)
    predictions, probabilities = _keras_binary_prediction(fitted['model'], X_test, BATCH_SIZE_OVERRIDES['kilichev_cnn'])
    return (make_binary_target(test_frame), predictions, probabilities, test_frame)


In [17]:
def fit_rf_selected_knn(train_frame, seed):
    set_all_seeds(seed)
    train_frame = downsample_frame(train_frame, MAX_TRAIN_ROWS, seed)
    X_all_df = train_frame[FEATURE_COLUMNS].copy()
    y_all = make_binary_target(train_frame)
    X_train_df, _, y_train, _ = train_test_split(X_all_df, y_all, test_size=0.3, random_state=seed, stratify=y_all, shuffle=True)
    numerical_columns = [column for column in FEATURE_COLUMNS if column not in CATEGORICAL_COLUMNS]
    preprocessor = ColumnTransformer(transformers=[('numerical', MinMaxScaler(feature_range=(0, 1), clip=True), numerical_columns), ('categorical', OneHotEncoder(handle_unknown='ignore', sparse_output=False, dtype=np.float32), CATEGORICAL_COLUMNS)], sparse_threshold=0.0)
    print('    Fitting preprocessing and Random Forest selector...', flush=True)
    X_train = preprocessor.fit_transform(X_train_df).astype(np.float32)
    random_forest = RandomForestClassifier(n_estimators=300, random_state=seed, n_jobs=-1)
    random_forest.fit(X_train, y_train)
    selected_mask = random_forest.feature_importances_ >= 0.037
    if not np.any(selected_mask):
        selected_mask[np.argmax(random_forest.feature_importances_)] = True
    knn = KNeighborsClassifier(n_neighbors=5, weights='uniform', metric='euclidean', n_jobs=-1)
    print('    Training KNN...', flush=True)
    knn.fit(X_train[:, selected_mask], y_train)
    return {'model': knn, 'preprocessor': preprocessor, 'selected_mask': selected_mask, 'seed': seed, 'n_train': len(train_frame)}

def predict_rf_selected_knn(fitted, test_frame):
    test_frame = _prepare_test_frame(test_frame, fitted['seed'])
    X_test = fitted['preprocessor'].transform(test_frame[FEATURE_COLUMNS]).astype(np.float32)
    X_test = X_test[:, fitted['selected_mask']]
    probabilities = fitted['model'].predict_proba(X_test)[:, 1]
    predictions = (probabilities >= 0.5).astype(np.int32)
    return (make_binary_target(test_frame), predictions, probabilities, test_frame)


In [18]:
class KANLinear(nn.Module):

    def __init__(self, in_features, out_features, grid_size=5, spline_order=3, grid_range=(-1.0, 1.0)):
        super().__init__()
        self.in_features = in_features
        self.out_features = out_features
        self.grid_size = grid_size
        self.spline_order = spline_order
        self.number_of_spline_coefficients = grid_size + spline_order
        grid_step = (grid_range[1] - grid_range[0]) / grid_size
        grid = torch.arange(-spline_order, grid_size + spline_order + 1, dtype=torch.float32) * grid_step + grid_range[0]
        grid = grid.expand(in_features, -1).contiguous()
        self.register_buffer('grid', grid)
        self.base_weight = nn.Parameter(torch.empty(out_features, in_features))
        self.spline_weight = nn.Parameter(torch.empty(out_features, in_features, self.number_of_spline_coefficients))
        self.spline_scaler = nn.Parameter(torch.empty(out_features, in_features))
        nn.init.kaiming_uniform_(self.base_weight, a=np.sqrt(5))
        nn.init.normal_(self.spline_weight, mean=0.0, std=0.02)
        nn.init.ones_(self.spline_scaler)

    def b_splines(self, x):
        x_expanded = x.unsqueeze(-1)
        grid = self.grid
        bases = ((x_expanded >= grid[:, :-1]) & (x_expanded < grid[:, 1:])).to(x.dtype)
        for order in range(1, self.spline_order + 1):
            left_numerator = x_expanded - grid[:, :-(order + 1)]
            left_denominator = grid[:, order:-1] - grid[:, :-(order + 1)]
            right_numerator = grid[:, order + 1:] - x_expanded
            right_denominator = grid[:, order + 1:] - grid[:, 1:-order]
            left_term = left_numerator / left_denominator * bases[:, :, :-1]
            right_term = right_numerator / right_denominator * bases[:, :, 1:]
            bases = left_term + right_term
        return bases.contiguous()

    def forward(self, x):
        base_output = F.linear(F.silu(x), self.base_weight)
        spline_bases = self.b_splines(x)
        scaled_spline_weights = self.spline_weight * self.spline_scaler.unsqueeze(-1)
        spline_output = F.linear(spline_bases.reshape(x.shape[0], -1), scaled_spline_weights.reshape(self.out_features, -1))
        return base_output + spline_output


In [19]:
def fit_ckan(train_frame, seed):
    set_all_seeds(seed)
    train_frame = downsample_frame(train_frame, MAX_TRAIN_ROWS, seed)
    X_all_df = train_frame[FEATURE_COLUMNS].copy()
    y_all = make_binary_target(train_frame)
    X_train_df, X_val_df, y_train, y_val = train_test_split(X_all_df, y_all, test_size=VALIDATION_SIZE, random_state=seed, stratify=y_all)
    numerical_columns = [column for column in FEATURE_COLUMNS if column not in CATEGORICAL_COLUMNS]
    preprocessor = ColumnTransformer(transformers=[('numerical', MinMaxScaler(feature_range=(0, 1), clip=True), numerical_columns), ('categorical', OneHotEncoder(handle_unknown='ignore', sparse_output=False, dtype=np.float32), CATEGORICAL_COLUMNS)], sparse_threshold=0.0)
    print('    Fitting CKAN preprocessing...', flush=True)
    X_train = preprocessor.fit_transform(X_train_df).astype(np.float32)
    X_val = preprocessor.transform(X_val_df).astype(np.float32)
    train_loader = DataLoader(TensorDataset(torch.from_numpy(X_train).unsqueeze(1), torch.from_numpy(y_train.astype(np.int64))), batch_size=BATCH_SIZE_OVERRIDES['ckan'], shuffle=True)
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = CKANBinaryDynamic().to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=0.001, weight_decay=0.01)
    print('    Training CKAN...', flush=True)
    for epoch in range(EPOCH_OVERRIDES['ckan']):
        model.train()
        running_loss = 0.0
        for batch_features, batch_labels in train_loader:
            batch_features = batch_features.to(device)
            batch_labels = batch_labels.to(device)
            optimizer.zero_grad()
            logits = model(batch_features)
            loss = criterion(logits, batch_labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * len(batch_labels)
        print(f'      epoch {epoch + 1}/{EPOCH_OVERRIDES['ckan']} loss={running_loss / len(train_loader.dataset):.6f}', flush=True)
    return {'model': model, 'preprocessor': preprocessor, 'device': device, 'seed': seed, 'n_train': len(train_frame)}

def predict_ckan(fitted, test_frame):
    test_frame = _prepare_test_frame(test_frame, fitted['seed'])
    X_test = fitted['preprocessor'].transform(test_frame[FEATURE_COLUMNS]).astype(np.float32)
    test_loader = DataLoader(TensorDataset(torch.from_numpy(X_test).unsqueeze(1), torch.from_numpy(make_binary_target(test_frame).astype(np.int64))), batch_size=BATCH_SIZE_OVERRIDES['ckan'], shuffle=False)
    fitted['model'].eval()
    predictions = []
    probabilities = []
    with torch.no_grad():
        for batch_features, _ in test_loader:
            batch_features = batch_features.to(fitted['device'])
            logits = fitted['model'](batch_features)
            probs = torch.softmax(logits, dim=1)
            predictions.extend(torch.argmax(probs, dim=1).cpu().numpy())
            probabilities.extend(probs[:, 1].cpu().numpy())
    return (make_binary_target(test_frame), np.asarray(predictions, dtype=np.int32), np.asarray(probabilities, dtype=np.float32), test_frame)


In [20]:
class CKANBinaryDynamic(nn.Module):

    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv1d(in_channels=1, out_channels=32, kernel_size=3, stride=1, padding=0, bias=True)
        self.conv2 = nn.Conv1d(in_channels=32, out_channels=64, kernel_size=3, stride=1, padding=0, bias=True)
        self.adaptive_pool = nn.AdaptiveAvgPool1d(output_size=1)
        self.kan = KANLinear(in_features=64, out_features=2, grid_size=5, spline_order=3, grid_range=(-1.0, 1.0))

    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))
        x = self.adaptive_pool(x).squeeze(-1)
        return self.kan(x)

def fit_predict_ckan(train_frame, test_frame, seed):
    set_all_seeds(seed)
    train_frame = downsample_frame(train_frame, MAX_TRAIN_ROWS, seed)
    test_frame = downsample_frame(test_frame, MAX_TEST_ROWS, seed)
    X_all_df = train_frame[FEATURE_COLUMNS].copy()
    X_test_df = test_frame[FEATURE_COLUMNS].copy()
    y_all = make_binary_target(train_frame)
    y_test = make_binary_target(test_frame)
    X_train_df, X_val_df, y_train, y_val = train_test_split(X_all_df, y_all, test_size=VALIDATION_SIZE, random_state=seed, stratify=y_all)
    numerical_columns = [column for column in FEATURE_COLUMNS if column not in CATEGORICAL_COLUMNS]
    preprocessor = ColumnTransformer(transformers=[('numerical', MinMaxScaler(feature_range=(0.0, 1.0), clip=True), numerical_columns), ('categorical', OneHotEncoder(handle_unknown='ignore', sparse_output=False, dtype=np.float32), CATEGORICAL_COLUMNS)], sparse_threshold=0.0)
    X_train = preprocessor.fit_transform(X_train_df).astype(np.float32)
    X_val = preprocessor.transform(X_val_df).astype(np.float32)
    X_test = preprocessor.transform(X_test_df).astype(np.float32)
    train_dataset = TensorDataset(torch.from_numpy(X_train).unsqueeze(1), torch.from_numpy(y_train.astype(np.int64)))
    validation_dataset = TensorDataset(torch.from_numpy(X_val).unsqueeze(1), torch.from_numpy(y_val.astype(np.int64)))
    test_dataset = TensorDataset(torch.from_numpy(X_test).unsqueeze(1), torch.from_numpy(y_test.astype(np.int64)))
    batch_size = BATCH_SIZE_OVERRIDES['ckan']
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    validation_loader = DataLoader(validation_dataset, batch_size=batch_size, shuffle=False)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = CKANBinaryDynamic().to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=0.001, weight_decay=0.01)
    for _ in range(EPOCH_OVERRIDES['ckan']):
        model.train()
        for batch_features, batch_labels in train_loader:
            batch_features = batch_features.to(device)
            batch_labels = batch_labels.to(device)
            optimizer.zero_grad()
            logits = model(batch_features)
            loss = criterion(logits, batch_labels)
            loss.backward()
            optimizer.step()
    model.eval()
    predictions = []
    probabilities = []
    with torch.no_grad():
        for batch_features, _ in test_loader:
            batch_features = batch_features.to(device)
            logits = model(batch_features)
            probs = torch.softmax(logits, dim=1)
            predictions.extend(torch.argmax(probs, dim=1).cpu().numpy())
            probabilities.extend(probs[:, 1].cpu().numpy())
    return (y_test, np.asarray(predictions, dtype=np.int32), np.asarray(probabilities, dtype=np.float32))


In [21]:
MODEL_REGISTRY = {'kwon_cnn': {'fit': fit_kwon, 'predict': predict_kwon}, 'vinayakumar_dnn': {'fit': fit_vinayakumar, 'predict': predict_vinayakumar}, 'kilichev_cnn': {'fit': fit_kilichev, 'predict': predict_kilichev}, 'rf_selected_knn': {'fit': fit_rf_selected_knn, 'predict': predict_rf_selected_knn}, 'ckan': {'fit': fit_ckan, 'predict': predict_ckan}}
unknown_models = sorted(set(MODEL_KEYS) - set(MODEL_REGISTRY))
if unknown_models:
    raise KeyError(f'Unknown MODEL_KEYS: {unknown_models}')
print('Selected models:', MODEL_KEYS)


Selected models: ['kwon_cnn', 'vinayakumar_dnn', 'kilichev_cnn', 'rf_selected_knn', 'ckan']


In [22]:
def _merge_and_save(path, new_frame, duplicate_columns):
    if path.exists():
        previous = pd.read_csv(path)
        combined = pd.concat([previous, new_frame], ignore_index=True)
    else:
        combined = new_frame.copy()
    combined = combined.drop_duplicates(subset=duplicate_columns, keep='last')
    combined.to_csv(path, index=False)
    return combined

def evaluate_model_on_test_sets(model_key, train_frame, test_sets, experiment, training_variant, seed):
    adapter = MODEL_REGISTRY[model_key]
    print(f'\n{experiment} | {model_key} | {training_variant} | seed={seed}', flush=True)
    print('  Training once...', flush=True)
    fit_started = time.perf_counter()
    fitted = adapter['fit'](train_frame, seed)
    fit_seconds = time.perf_counter() - fit_started
    print(f'  Training completed in {fit_seconds / 60:.2f} minutes.', flush=True)
    overall_rows = []
    category_parts = []
    pooled_true = []
    pooled_prediction = []
    pooled_score = []
    for test_source, test_frame in test_sets.items():
        print(f'  Testing {test_source} ({len(test_frame):,} raw rows)...', flush=True)
        prediction_started = time.perf_counter()
        y_true, y_pred, y_score, evaluated_test = adapter['predict'](fitted, test_frame)
        prediction_seconds = time.perf_counter() - prediction_started
        metrics = calculate_binary_metrics(y_true, y_pred, y_score)
        evaluated_attack_fraction = float(np.mean(y_true == 1))
        overall_rows.append({'experiment': experiment, 'training_variant': training_variant, 'model': model_key, 'seed': seed, 'test_source': test_source, 'n_train_raw': int(len(train_frame)), 'fit_seconds': fit_seconds, 'prediction_seconds': prediction_seconds, 'test_attack_pct': 100.0 * evaluated_attack_fraction, 'test_normal_pct': 100.0 * (1.0 - evaluated_attack_fraction), **metrics})
        categories = category_recall_table(evaluated_test, y_true, y_pred)
        categories['experiment'] = experiment
        categories['training_variant'] = training_variant
        categories['model'] = model_key
        categories['seed'] = seed
        categories['test_source'] = test_source
        category_parts.append(categories)
        pooled_true.append(y_true)
        pooled_prediction.append(y_pred)
        pooled_score.append(y_score)
        print(f'    evaluated={len(y_true):,}, accuracy={metrics['accuracy']:.4f}, recall={metrics['recall']:.4f}, f1={metrics['f1']:.4f}', flush=True)
    pooled_true = np.concatenate(pooled_true)
    pooled_prediction = np.concatenate(pooled_prediction)
    pooled_score = np.concatenate(pooled_score)
    pooled_metrics = calculate_binary_metrics(pooled_true, pooled_prediction, pooled_score)
    pooled_attack_fraction = float(np.mean(pooled_true == 1))
    overall_rows.append({'experiment': experiment, 'training_variant': training_variant, 'model': model_key, 'seed': seed, 'test_source': 'POOLED_ALL_ROWS', 'n_train_raw': int(len(train_frame)), 'fit_seconds': fit_seconds, 'prediction_seconds': sum((row['prediction_seconds'] for row in overall_rows)), 'test_attack_pct': 100.0 * pooled_attack_fraction, 'test_normal_pct': 100.0 * (1.0 - pooled_attack_fraction), **pooled_metrics})
    del fitted
    gc.collect()
    tf.keras.backend.clear_session()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return (pd.DataFrame(overall_rows), pd.concat(category_parts, ignore_index=True))

def run_experiment(experiment, train_variants, test_sets):
    overall_path = RESULTS_DIR / f'{experiment}_all_runs_partial.csv'
    category_path = RESULTS_DIR / f'{experiment}_category_runs_partial.csv'
    completed = pd.read_csv(overall_path) if overall_path.exists() else pd.DataFrame()
    overall_accumulated = completed.copy()
    category_accumulated = pd.read_csv(category_path) if category_path.exists() else pd.DataFrame()
    expected_sources = set(test_sets) | {'POOLED_ALL_ROWS'}
    for model_key in MODEL_KEYS:
        for training_variant, train_frame in train_variants.items():
            for seed in SEEDS:
                if not completed.empty:
                    mask = completed['experiment'].eq(experiment) & completed['model'].eq(model_key) & completed['training_variant'].eq(training_variant) & completed['seed'].eq(seed)
                    completed_sources = set(completed.loc[mask, 'test_source'].astype(str))
                else:
                    completed_sources = set()
                if expected_sources.issubset(completed_sources):
                    print(f'SKIP completed: {model_key}, {training_variant}, seed={seed}', flush=True)
                    continue
                overall, categories = evaluate_model_on_test_sets(model_key=model_key, train_frame=train_frame, test_sets=test_sets, experiment=experiment, training_variant=training_variant, seed=seed)
                overall_accumulated = _merge_and_save(overall_path, overall, ['experiment', 'training_variant', 'model', 'seed', 'test_source'])
                category_accumulated = _merge_and_save(category_path, categories, ['experiment', 'training_variant', 'model', 'seed', 'test_source', 'test_category'])
                completed = overall_accumulated
    return (overall_accumulated, category_accumulated)


In [23]:
def summarize_overall(results):
    metrics = ['accuracy', 'balanced_accuracy', 'precision', 'recall', 'specificity', 'f1', 'macro_f1', 'mcc', 'false_positive_rate', 'roc_auc']
    return results.groupby(['experiment', 'training_variant', 'model', 'test_source'], dropna=False)[metrics].agg(['mean', 'std', 'min', 'max']).reset_index()

def summarize_categories(results):
    return results.groupby(['experiment', 'training_variant', 'model', 'test_source', 'test_category'], dropna=False).agg(support=('support', 'first'), binary_recall_mean=('binary_recall', 'mean'), binary_recall_std=('binary_recall', 'std'), benign_specificity_mean=('benign_specificity', 'mean'), benign_specificity_std=('benign_specificity', 'std')).reset_index()

def source_macro_summary(results):
    source_only = results[results['test_source'].ne('POOLED_ALL_ROWS')]
    by_seed = source_only.groupby(['experiment', 'training_variant', 'model', 'seed']).agg(source_macro_accuracy=('accuracy', 'mean'), source_macro_balanced_accuracy=('balanced_accuracy', 'mean'), source_macro_f1=('f1', 'mean'), source_macro_mcc=('mcc', 'mean'), source_macro_recall=('recall', 'mean'), source_macro_specificity=('specificity', 'mean')).reset_index()
    return by_seed.groupby(['experiment', 'training_variant', 'model']).agg(['mean', 'std', 'min', 'max']).reset_index()

def save_final_results(experiment, overall, categories):
    overall_summary = summarize_overall(overall)
    category_summary = summarize_categories(categories)
    source_summary = source_macro_summary(overall)
    result_columns = ['model', 'seed', 'test_source', 'n_train_raw', 'fit_seconds', 'prediction_seconds', 'test_attack_pct', 'test_normal_pct', 'n_test', 'accuracy', 'balanced_accuracy', 'precision', 'recall', 'specificity', 'f1', 'macro_f1', 'mcc', 'false_positive_rate', 'tn', 'fp', 'fn', 'tp', 'roc_auc']
    source_order = ['cic2017', 'cic2018', 'darpa', 'unsw-nb15', 'POOLED_ALL_ROWS']
    table_results = overall.copy()
    table_results['__model'] = pd.Categorical(table_results['model'], categories=MODEL_KEYS, ordered=True)
    table_results['__seed'] = pd.Categorical(table_results['seed'], categories=SEEDS, ordered=True)
    table_results['__source'] = pd.Categorical(table_results['test_source'], categories=source_order, ordered=True)
    table_results = table_results.sort_values(['__model', '__seed', '__source']).loc[:, result_columns]
    expected_rows = len(MODEL_KEYS) * len(SEEDS) * len(source_order)
    if len(table_results) != expected_rows:
        raise AssertionError(f'Expected {expected_rows} result rows, got {len(table_results)}.')
    table_results.to_csv(RESULTS_DIR / 'all_runs_NSL-KDD.csv', index=False)
    overall.to_csv(RESULTS_DIR / f'{experiment}_all_runs_with_metadata.csv', index=False)
    categories.to_csv(RESULTS_DIR / f'{experiment}_category_runs.csv', index=False)
    overall_summary.to_csv(RESULTS_DIR / f'{experiment}_summary.csv', index=False)
    category_summary.to_csv(RESULTS_DIR / f'{experiment}_category_summary.csv', index=False)
    source_summary.to_csv(RESULTS_DIR / f'{experiment}_source_macro_summary.csv', index=False)
    return (overall_summary, category_summary, source_summary)


In [ ]:
exp4_overall, exp4_categories = run_experiment(experiment=EXP4_EXPERIMENT, train_variants={'mixed_source_balanced_65120': exp4_train_balanced}, test_sets=exp4_test_sets)
exp4_summary, exp4_category_summary, exp4_source_macro_summary = save_final_results(EXP4_EXPERIMENT, exp4_overall, exp4_categories)
exp4_test_ratio_manifest.to_csv(RESULTS_DIR / f'{EXP4_EXPERIMENT}_test_ratio_manifest.csv', index=False)


mixed_source_nsl_kdd | kwon_cnn | mixed_source_balanced_65120 | seed=11
  Training once...
    Fitting preprocessing...
    Training Kwon CNN...
Epoch 1/10
2035/2035 ━━━━━━━━━━━━━━━━━━━━ 14s 3ms/step - accuracy: 0.9241 - loss: 0.2021
Epoch 2/10
2035/2035 ━━━━━━━━━━━━━━━━━━━━ 8s 4ms/step - accuracy: 0.9452 - loss: 0.1518
Epoch 3/10
2035/2035 ━━━━━━━━━━━━━━━━━━━━ 8s 3ms/step - accuracy: 0.9501 - loss: 0.1417
Epoch 4/10
2035/2035 ━━━━━━━━━━━━━━━━━━━━ 7s 3ms/step - accuracy: 0.9520 - loss: 0.1360
Epoch 5/10
2035/2035 ━━━━━━━━━━━━━━━━━━━━ 6s 3ms/step - accuracy: 0.9529 - loss: 0.1336
Epoch 6/10
2035/2035 ━━━━━━━━━━━━━━━━━━━━ 6s 3ms/step - accuracy: 0.9549 - loss: 0.1292
Epoch 7/10
2035/2035 ━━━━━━━━━━━━━━━━━━━━ 6s 3ms/step - accuracy: 0.9572 - loss: 0.1242
Epoch 8/10
2035/2035 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - accuracy: 0.9572 - loss: 0.1249
Epoch 9/10
2035/2035 ━━━━━━━━━━━━━━━━━━━━ 6s 3ms/step - accuracy: 0.9579 - loss: 0.1224
Epoch 10/10
2035/2035 ━━━━━━━━━━━━━━━━━━━━ 6s 3ms/step - accu